# SymboLM (v2.0) — Stage 5: Scratchpad 2.0 & Unbounded Parsimony GRPO

**Author & Principal Architect:** Sanish Gyawali  
**AI Systems Exoskeleton:** Antigravity (Google DeepMind)  
**Hardware:** Kaggle NVIDIA Tesla T4 GPU (CUDA)  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

---  
### Core Stage 5 Objectives:
1. **Unbounded Parsimony Gradient:** No artificial token ceilings. Monotonically rewards using the least tokens possible ($R \propto \frac{K}{K + |\tau|}$).
2. **Elastic Scratchpad 2.0:** Dynamic depth allocation across the Cognitive Depth Spectrum ($\Delta_0 \to \Delta_3$) with up to 192 tokens for deep proofs.
3. **Latent Knowledge Leverage:** Discourages verbal recitation of known axioms, pushing the policy toward Minimum Description Length (MDL).
4. **Warm-Start from Stage 4:** Inherits 1.0% syntax noise and clean anti-repetition termination.

In [ ]:
# Cell 1: Environment & Hardware Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Warning] No CUDA GPU found!')


In [ ]:
# Cell 2: Install required libraries
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q transformers peft trl accelerate sympy datasets huggingface_hub


In [ ]:
# Cell 3: Clone latest SymboLM repository with Stage 5 parsimony curriculum
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
print('[OK] Successfully cloned latest SymboLM repository!')


In [ ]:
# Cell 4: Mount Stage 4 Efficiency Adapter from Kernel Source
import os
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/stage4_efficient_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# 1. Search for stage4_adapter in mounted kernel inputs
for p in Path('/kaggle/input').glob('**/stage4_adapter'):
    if (p / 'adapter_config.json').exists():
        print(f'[OK] Found Stage 4 adapter at: {p}')
        for item in p.iterdir():
            if item.is_file():
                shutil.copy2(item, target_dir / item.name)
        found = True
        break

# 2. Search for tar archive
if not found:
    for arc in Path('/kaggle/input').glob('**/*stage4*policy*.tar.gz'):
        print(f'[OK] Found Stage 4 tar archive at: {arc}. Unpacking...')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

print('[Status] Validating Stage 4 Warm-Start Files:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')

assert (target_dir / 'adapter_config.json').exists(), 'CRITICAL: adapter_config.json missing!'
assert (target_dir / 'adapter_model.safetensors').exists(), 'CRITICAL: adapter_model.safetensors missing!'
print('[VERIFIED] Stage 4 adapter ready as Stage 5 warm-start!')


In [ ]:
# Cell 5: Run Stage 5 Parsimony & Scratchpad 2.0 GRPO Training
!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/symboLM/training/stage5_parsimony_train.py \
  --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
  --warmstart_adapter /kaggle/working/symboLM/checkpoints/stage4_efficient_adapter \
  --output_dir /kaggle/working/stage5_parsimony_adapter \
  --data_file /kaggle/working/symboLM/data/stage5_parsimony_curriculum.jsonl \
  --learning_rate 1e-5 \
  --max_steps 150 \
  --max_completion_length 192 \
  --num_generations 4 \
  --beta 0.04 \
  --temperature 0.3


In [ ]:
# Cell 6: Post-Training Multi-Depth Verification Audit
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print('=== STAGE 5 MULTI-DEPTH INLINE AUDIT ===')
adapter_dir = Path('/kaggle/working/stage5_parsimony_adapter')
tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir), trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    torch_dtype=torch.float16,
    device_map={'': 'cuda:0'},
    trust_remote_code=True
)
base_model.resize_token_embeddings(151712)
model = PeftModel.from_pretrained(base_model, str(adapter_dir))
model.eval()

test_suite = [
    ('Δ0 Atomic Fact', 'What is the chemical symbol for Potassium?', 'K'),
    ('Δ1 Linear Invariant', 'If x + 1/x = 6, find the exact value of x^2 + 1/x^2.', '34'),
    ('Δ2 Geometry Invariant', 'A right triangle has inradius r = 6 and perimeter P = 72. Find its area using Area = r * s.', '216'),
    ('Δ3 Deep Congruence', 'Solve for x in [0, 76]: x ≡ 4 (mod 7) and x ≡ 9 (mod 11). Find x.', '53'),
]

for depth_label, q, expected in test_suite:
    prompt = f'<｜User｜>{q}<｜Assistant｜><think>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda:0')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=False)
    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=False)
    toks = len(gen.split())
    print(f'\n[{depth_label}] Target: {expected} | Emitted: {toks} tokens')
    print('Generation Preview:')
    print(gen[:300])
    print('-' * 70)


In [ ]:
# Cell 7: Package & Export Stage 5 Final Policy
print('[Export] Archiving Stage 5 Aligned Policy Adapter...')
!tar -czf /kaggle/working/symboLM_stage5_final_policy.tar.gz -C /kaggle/working/stage5_parsimony_adapter .
sz = os.path.getsize('/kaggle/working/symboLM_stage5_final_policy.tar.gz') / 1e6
print(f'[OK] Successfully exported /kaggle/working/symboLM_stage5_final_policy.tar.gz ({sz:.2f} MB)')
